In [1]:
import pandas as pd
from collections import Counter
import os

raw_data_customers= r"C:\Workspace-Airflow\Mini_Project_1\input_data\customers.csv"
raw_data_orders= r"C:\Workspace-Airflow\Mini_Project_1\input_data\orders.csv"
raw_data_products= r"C:\Workspace-Airflow\Mini_Project_1\input_data\products.csv"
raw_data_clickstream= r"C:\Workspace-Airflow\Mini_Project_1\input_data\clickstream.csv"
df_customers= pd.read_csv(raw_data_customers)
df_orders= pd.read_csv(raw_data_orders)
df_products= pd.read_csv(raw_data_products)
df_clickstream= pd.read_csv(raw_data_clickstream)

df_customers
df_orders
df_products
df_clickstream

#validate the schema consistency

df_customers.dtypes
df_orders.dtypes
df_products.dtypes
df_clickstream.dtypes

df_customers.columns
df_orders.columns
df_products.columns
df_clickstream.columns

df_customers['customer_id'].is_unique
df_orders['order_id'].is_unique
df_products['product_id'].is_unique
df_clickstream['event_timestamp'].is_unique

df_customers.isnull().sum()
df_orders.isnull().sum()
df_products.isnull().sum()
df_clickstream.isnull().sum()


#detect duplicate customers

print(df_customers[df_customers['customer_id'].duplicated(keep=False)])

#identify invalid records
df_invalid_customers= df_customers.isnull()
df_invalid_customers

df_invalid_orders = df_orders.isnull()
df_invalid_orders

df_invalid_products = df_products.isnull()
df_invalid_products

df_invalid_clickstream = df_clickstream.isnull()
df_invalid_clickstream


df_orders[df_orders['unit_price'] < 0]
df_orders[df_orders['quantity'] < 0]
df_products[df_products['cost_price'] < 0]
df_products[df_products['selling_price'] < 0]


#standardize city names
df_customers['city'] = df_customers['city'].str.title()
df_customers['city']

#convert timestamp correctly

df_clickstream['event_timestamp'] = pd.to_datetime(df_clickstream['event_timestamp'], format = 'mixed' , dayfirst=True, utc=True, errors='coerce')
df_clickstream['event_timestamp']

#handle missing values
df_customers.fillna("NA",inplace=True)
df_orders.fillna({'unit_price': 0, 'quantity': 0}, inplace=True)
df_products.fillna({'cost_price': 0, 'selling_price': 0}, inplace=True)
df_clickstream.fillna({'event_type': 'NA'}, inplace=True)

#remove corrupted rows

df_orders = df_orders[(df_orders['unit_price'] >= 0) & (df_orders['quantity'] >= 0)]
df_products = df_products[(df_products['cost_price'] >= 0) & (df_products['selling_price'] >= 0)]
df_customers = df_customers.drop_duplicates(subset='customer_id', keep='first')
df_customers = df_customers.dropna()
df_clickstream = df_clickstream.drop_duplicates(subset='event_timestamp', keep='first')
df_clickstream = df_clickstream.dropna()


#most visited pages

unique_pages = set(df_clickstream['page_url'])
print(unique_pages)
page_count = Counter(df_clickstream['page_url'])
print(page_count)
most_visited_pages = page_count.most_common(1)[0]
print(f"The most visited page is : {most_visited_pages[0]}")
print(f"The most visit count is : {most_visited_pages[1]}")
    
#calculate session counts



# Sort by user and timestamp
df_clickstream = df_clickstream.sort_values(['customer_id', 'event_timestamp'])

# Time difference between events
df_clickstream['time_diff'] = df_clickstream.groupby('customer_id')['event_timestamp'].diff()

# New session if gap > 30 minutes
df_clickstream['new_session'] = df_clickstream['time_diff'] > pd.Timedelta(minutes=30)

# Assign session IDs
df_clickstream['session_id'] = df_clickstream.groupby('customer_id')['new_session'].cumsum()

# Count sessions per user
session_counts = df_clickstream.groupby('customer_id')['session_id'].nunique()
print(session_counts)

print("Total sessions:", df_clickstream['session_id'].nunique())



#Find bounce rate



# Pages per session
pages_per_session = df_clickstream.groupby(['customer_id', 'session_id'])['page_url'].count()

# Bounced sessions = sessions with only 1 page
bounced_sessions = pages_per_session == 1

# Number of bounced sessions
num_bounced_sessions = bounced_sessions.sum()

# Total sessions
total_sessions = len(pages_per_session)

# Bounce rate
bounce_rate = num_bounced_sessions / total_sessions
print("Bounce rate:", bounce_rate)




#Find mobile vs desktop traffic percentage


def classify_device_type(device_type):
    device_type = device_type.lower()
    if device_type == 'mobile':
        return 'Mobile'
    elif device_type == 'desktop':
        return 'Desktop'
    else:
        return 'Other'

df_clickstream['device_type'] = df_clickstream['device_type'].apply(classify_device_type)
device_counts = df_clickstream['device_type'].value_counts()
total_visits = device_counts.sum()
device_percentage = (device_counts / total_visits) * 100

print(f"Device traffic percentage between Mobile and Desktop is {device_percentage['Mobile']:.2f}% and {device_percentage['Desktop']:.2f}%", device_percentage)

#Export analytical data to a new csv file and parquet file

df_clickstream.to_csv(r"C:\Workspace-Airflow\Mini_Project_1\Output_data\clickstream_cleaned.csv", index=False)
df_clickstream.to_parquet(r"C:\Workspace-Airflow\Mini_Project_1\Output_data\clickstream_cleaned.parquet", index=False)
#df_clickstream = pd.read_parquet(r"C:\Workspace-Airflow\Mini_Project_1\Output_data\clickstream_cleaned.parquet")









Empty DataFrame
Columns: [customer_id, customer_name, email, city, signup_date]
Index: []
{'/help', '/checkout', '/search', '/account', '/cart', '/home', '/product/101', '/products', '/orders', '/product/105'}
Counter({'/search': 161, '/products': 159, '/orders': 156, '/product/105': 155, '/home': 152, '/product/101': 146, '/checkout': 142, '/account': 137, '/cart': 134, '/help': 122})
The most visited page is : /search
The most visit count is : 161
customer_id
1.0      12
2.0      17
3.0      12
4.0      12
5.0       8
         ..
96.0     11
97.0     17
98.0      8
99.0     15
100.0    12
Name: session_id, Length: 100, dtype: int64
Total sessions: 24
Bounce rate: 0.9986320109439124
Device traffic percentage between Mobile and Desktop is 42.83% and 28.42% device_type
Mobile     42.827869
Other      28.756831
Desktop    28.415301
Name: count, dtype: float64
